# Gateway tracking with an uncertain halo observer

Compare Gateway orbit estimation over one observer period using (1) exact observer knowledge and (2) an observer orbit estimated from lunar surface observations. Both cases use the same truth trajectories, Gateway prior and noisy Gateway measurements.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "configs/baseline.json").exists() and (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from within the project checkout")
sys.path.insert(0, str(ROOT / "src"))
from halo_tracking.study import load_config
cfg = load_config(ROOT / "configs/baseline.json")
cfg["simulation"]["periods"] = 1.0  # One nominal observer period


## 1. Matched comparison

| Assumption | Perfect observer | LSO-estimated observer |
|---|---|---|
| Observer state supplied to Gateway estimator | Exact true state | State estimated from LSO angles |
| Initial observer uncertainty | Zero | 1 km and 0.01 m/s per axis, 1σ |
| Initial Gateway uncertainty | 1 km and 0.01 m/s per axis, 1σ | Same |
| Observer-to-Gateway camera | 1 arcsecond per angle | Same measurements and noise |
| LSO | Not needed | 30° N at 270° E and 90° E; 85° S at 90° E; 1 arcsecond per angle |

These are the baseline settings; the configuration table below displays the values actually used. Both optical links use **angles only** in this focused comparison. The cadence is 30 minutes. Earth/Moon occultation is included for Gateway observations; the LSO uses the configured elevation mask. Optical brightness, Sun exclusions and camera attitude errors are not modeled.

The true observer remains on the supplied initial-value trajectory in both cases. The imperfect case introduces uncertainty in our **knowledge** of it, not a different physical orbit.

In [ ]:
from IPython.display import HTML, Markdown, display
settings = [
    ("Duration", f"1 observer nominal period = {cfg['orbit']['period_tu'] * cfg['time_unit_s'] / 86400:.2f} days"),
    ("Observer initial position / velocity 1σ", f"{cfg['estimation']['initial_position_sigma_km']:g} km / {1000*cfg['estimation']['initial_velocity_sigma_km_s']:g} m/s per axis"),
    ("Gateway initial position / velocity 1σ", f"{cfg['comparison']['gateway_position_sigma_km']:g} km / {1000*cfg['comparison']['gateway_velocity_sigma_km_s']:g} m/s per axis"),
    ("LSO angular noise 1σ", f"{cfg['measurement']['angle_sigma_arcsec']:g} arcsec per angle"),
    ("Gateway camera angular noise 1σ", f"{cfg['comparison']['gateway_camera_sigma_arcsec']:g} arcsec per angle")]
display(HTML("<table>"+"".join(f"<tr><th>{name}</th><td>{value}</td></tr>" for name,value in settings)+"</table>"))

In [ ]:
from halo_tracking.gateway_comparison import run_gateway_comparison, save_comparison
result = run_gateway_comparison(cfg)
report = save_comparison(result, ROOT / "results/gateway_comparison")

## 2. Geometry and what the LSO knows

The first figure spans exactly one observer nominal period (Gateway repeats at its own period). It is context: both true trajectories are identical across the two estimation cases. The dots mark their initial states. The axis scale is in thousands of kilometers, with equal physical scale on all three axes.

The second figure shows the observer's estimated-state error and its local covariance prediction. The shaded intervals contain observations from at least one LSO. The three-site network is evaluated at every scheduled epoch. **RSS σ is a covariance summary, not a guaranteed bound on the error magnitude.**

In [ ]:
from halo_tracking.plotting import geometry, observer_knowledge, gateway_comparison_figures, state_histories, export
fig_geometry = geometry(result)
export(fig_geometry, ROOT / "results/gateway_comparison/geometry.png")

In [ ]:
fig_observer = observer_knowledge(result)
export(fig_observer, ROOT / "results/gateway_comparison/observer.png")

## Observer position, velocity and acceleration

The solid curve is the perfect observer state (truth); the dashed curve is the LSO-predicted state. Components use Moon-centered inertial axes. These curves nearly overlap at orbit scale; the error plot above resolves their differences.

In [ ]:
fig_observer_states = state_histories(result, 'observer')
export(fig_observer_states, ROOT / 'results/gateway_comparison/observer_states.png')

## Gateway position, velocity and acceleration

Compare the true Gateway trajectory with the two tracked trajectories: one uses the perfect observer and one uses the LSO-predicted observer. Their small separation is quantified by the error plot and RMSE table below.

In [ ]:
fig_gateway_states = state_histories(result, 'gateway')
export(fig_gateway_states, ROOT / 'results/gateway_comparison/gateway_states.png')

## 3. What changes in Gateway tracking

**Actual-error plot:** distance between the estimated and true Gateway state in this one shared noise realization. Lower is better. It is an actual orbit estimate, not just a pointing-error projection.

**Predicted-uncertainty plot:** square root of the position/velocity covariance-block trace. Lower means the estimator predicts better knowledge. These are local linearized predictions; they can be optimistic if the approximation is inaccurate.

In the imperfect case, the estimator carries observer–Gateway cross-covariance through time. Gateway measurements do not refine the observer: its state knowledge comes from the LSO. LSO updates can also correct Gateway indirectly through the retained correlation.

In [ ]:
figures = gateway_comparison_figures(result)
fig_errors = figures["errors"]
export(fig_errors, ROOT / "results/gateway_comparison/errors.png")
import matplotlib.pyplot as plt
plt.close(figures["uncertainty"])

In [ ]:
figures_uncertainty = gateway_comparison_figures(result)
plt.close(figures_uncertainty["errors"])
fig_uncertainty = figures_uncertainty["uncertainty"]
export(fig_uncertainty, ROOT / "results/gateway_comparison/uncertainty.png")

## 4. Numerical conclusions

The table separates **actual errors from one realization** from **predicted uncertainty**. It should be read with the assumptions above, especially the Gateway prior and observing geometry.

In [ ]:
rows=[]
for key,label in [("perfect","Perfect observer knowledge"),("lso_estimated","Observer estimated from LSO")]:
    item=report[key]
    values=[label, f"{1000*item['position_rmse_km']:.1f}", f"{item['velocity_rmse_m_s']:.4f}",
            f"{1000*item['final_position_error_km']:.1f}",
            f"{1000*item['final_position_forecast_rss_sigma_km']:.1f}",
            f"{item['final_velocity_error_m_s']:.3g}"]
    rows.append("<tr>"+"".join("<td>"+v+"</td>" for v in values)+"</tr>")
headers=["Observer knowledge", "Gateway position RMSE over arc (m)", "Gateway velocity RMSE over arc (m/s)", "Final actual position error (m)", "Final predicted position RSS σ (m)", "Final actual velocity error (m/s)"]
display(HTML("<table><tr>"+"".join("<th>"+v+"</th>" for v in headers)+"</tr>"+"".join(rows)+"</table>"))

In [ ]:
import numpy as np
ideal=report["perfect"]; estimated=report["lso_estimated"]
ratio=estimated["final_position_forecast_rss_sigma_km"]/ideal["final_position_forecast_rss_sigma_km"]
available=np.flatnonzero(result["lso_available"])
start=result["times"][available[0]]*cfg["time_unit_s"]/86400 if len(available) else float("nan")
end=result["times"][available[-1]]*cfg["time_unit_s"]/86400 if len(available) else float("nan")
conclusions = f"""
- Final predicted Gateway position RSS σ: **{1000*ideal['final_position_forecast_rss_sigma_km']:.1f} m** with exact observer knowledge versus **{1000*estimated['final_position_forecast_rss_sigma_km']:.1f} m** with LSO-estimated knowledge ({ratio:.1f}×).
- Final actual position errors: **{1000*ideal['final_position_error_km']:.1f} m** versus **{1000*estimated['final_position_error_km']:.1f} m** in this realization.
- The network supplies **{int(result['sensor_visible'].sum())} site observations** across **{int(result['lso_available'].sum())} scheduled epochs**. Overlapping sites provide separate measurements.
- These EKF results illustrate ideal versus measured state knowledge. They are a benchmark, not an identified model or an ensemble accuracy claim.
"""
display(Markdown(conclusions))

## Model and limitations

Propagation uses the Earth–Moon CR3BP. The plots transform normalized rotating states into Moon-centered inertial position, velocity and acceleration. Acceleration is calculated from each state using the dynamics; it is not independently estimated.

The imperfect case retains observer–Gateway cross-covariance. LSO measurements update observer knowledge; Gateway measurements update Gateway without refining the observer. Initial covariances are diagonal, with zero observer–Gateway cross-covariance. The forecast uses zero innovations along the nominal trajectories; the actual estimate uses noisy measurements.

The supplied `SNRHL2` state is a southern NRHO initial condition rounded to four decimals. Its one-period closure miss is about 99 km. This study assumes no stationkeeping or unmodeled forces. Local filter covariance can be overconfident near nonlinear updates, so predicted uncertainty is a conditional comparison rather than a validated error bound.

Gateway's initial state comes from [the earlier cislunar project](https://github.com/scottblender/Gradient-Free-Optimization-for-Cislunar-SDA-2/blob/main/scripts/build_target_case_database.m). Both phases start at model time zero, without a mission epoch. Both trajectories use LU = 384400 km and TU = 375190.25852 s; the source Gateway project used TU = 375695 s.

## Uncontrolled observer departure from a corrected periodic reference

The physical observer truth starts at the original four-decimal SNRHL2 catalog state and is propagated continuously without station-keeping. **Only the nominal reference** is differentially corrected to a nearby periodic CR3BP orbit. All 20 horizons use that corrected nominal period, \(T\), so the observer's starting state and the reference are intentionally not identical.

The catalog's rounded state would miss closure by roughly 99 km after its supplied period, but the corrected nominal reference closes to a much tighter numerical tolerance. The first-period RMS position error is therefore **nonzero**, reflecting the initial difference between truth and nominal; this is expected, not a failed solver or a reference reset.

Position departure means distance between the continuous observer truth and its equal-phase periodic nominal position. It is **not** distance to Gateway, and a large departure is not by itself a validated station-keeping deadline. The ensemble shows sensitivity to additional seeded 1-km position and 0.01-m/s velocity perturbations.

In [ ]:
from halo_tracking.departure import reference_departure, save_departure
from halo_tracking.plotting import departure_figures
departure = reference_departure(cfg, periods=20, ensemble_count=8)
save_departure(departure, cfg, ROOT / 'results/departure')
departure_plots = departure_figures(departure, cfg)
export(departure_plots['departure'], ROOT / 'results/departure/departure.png')
export(departure_plots['period_growth'], ROOT / 'results/departure/period_growth.png')

In [ ]:
closure = departure['diagnostics']
periods = [2, 4, 6, 8, 10, 20]
rows = ''.join(f"<tr><td>{k}</td><td>{departure['period_position_rms_km'][k-1]:,.1f}</td>"
               f"<td>{departure['period_velocity_rms_m_s'][k-1]:,.3f}</td></tr>" for k in periods)
display(HTML('<table><tr><th>Period</th><th>RMS position departure (km)</th>'
             '<th>RMS velocity departure (m/s)</th></tr>' + rows + '</table>'))
display(Markdown(f"""
The corrected **nominal** one-period closure is **{closure['position_closure_km']:.6f} km** and **{closure['velocity_closure_m_s']:.6f} m/s**. The original rounded **catalog** state's one-period closure error is **{closure['catalog_closure_km']:.1f} km**. The initial rounded truth-to-nominal position offset is **{closure['initial_nominal_offset_km']:.1f} km**.

Rapid divergence is evaluated from this particular rounded-reference simulation and the 1 km / 0.01 m/s initial perturbation ensemble. Do not interpret transient decreases as stabilization, or set an operational cutoff before defining position/velocity tolerances.

The next stationkeeping decision is an acceptable departure corridor and prediction horizon. These curves support choosing that requirement; they do not establish the latest safe maneuver time by themselves.
"""))

## Cumulative uncontrolled observer trajectories

All grid panels show increasingly long prefixes of **the same uninterrupted propagation from the original rounded catalog state**: \(0\to T\), \(0\to2T\), ..., \(0\to20T\), where \(T\) is the corrected nominal halo period.

The dashed gray reference is a *differentially corrected periodic nominal trajectory*, repeated at each phase. **It is not the spacecraft's truth trajectory.** The first panel can therefore already show an offset between the truth and the reference. Each panel reports the maximum nominal-reference departure reached *any time up to that horizon*.

The first grid auto-scales each panel to show the entire growing trajectory, and a second grid holds the corrected nominal reference axis limits fixed. All panels have the same initial truth state; no restarts or impulses are applied.

In [ ]:
from halo_tracking.plotting import period_trajectory_grid
fig_period_grid = period_trajectory_grid(departure, cfg, scale_mode='local')
export(fig_period_grid, ROOT / 'results/departure/observer_period_grid.png')
display(fig_period_grid)
plt.close(fig_period_grid)

fig_reference_grid = period_trajectory_grid(departure, cfg, scale_mode='reference')
export(fig_reference_grid, ROOT / 'results/departure/observer_period_grid_fixed_scale.png')
plt.close(fig_reference_grid)

## Gateway tracking versus cumulative observer-period horizon

The Gateway study performs **one continuous simulation and estimator run**, starting from the original rounded observer initial condition and using the corrected nominal period for cumulative horizons. For each `k`, the row summarizes **all samples from `0 → kT`**, not just the kth period. There are no filter or covariance resets.

**Interpret these distances separately:**
- **Observer departure** is the observer's difference from the nominal one-period halo reference at matching phase. It is not distance traveled, distance from the Moon, or range to Gateway. The reference is differentially corrected to close periodically; the freely propagated physical observer still starts at the rounded catalog state.
- **Observer–Gateway range** is the actual Euclidean separation between the two spacecraft, relevant to the optical detection-range cutoff.

The table reports cumulative position/velocity RMSE, cumulative accepted Gateway and lunar-surface observations, **new observations in the most recent period**, and terminal predictive position uncertainty. If new observations are zero, the estimate is simply propagating its prior model; low cumulative RMSE does **not** mean Gateway is currently observable.

Measurements are rejected using geometric occultation, the configured 20° Sun / 10° Earth center-referenced keep-outs, and illustrative 250,000-km range caps. The Sun direction is a simplified model, not a calibrated ephemeris or instrument detection model.

Saved CSV rows are reused only if the configuration, requested horizon, and **cumulative corrected-nominal-reference version** all match; prior per-period files are invalidated.

The display uses **two tables with one row per cumulative horizon**: geometry and accepted measurements first, followed by Gateway estimation. The two observer-knowledge cases appear side by side in the second table. “Latest / total” observation counts show recent visibility loss even when cumulative totals remain large.

In [ ]:
import os
from halo_tracking.gateway_comparison import (
    run_gateway_period_study, save_gateway_period_study, load_gateway_period_study
)

observer_period_count = int(os.environ.get('HALO_GATEWAY_PERIODS', '20'))
period_output = ROOT / 'results/departure'
gateway_period_study = load_gateway_period_study(
    cfg, period_output, periods=observer_period_count)
if gateway_period_study is None:
    gateway_period_study = run_gateway_period_study(
        cfg, periods=observer_period_count)
    save_gateway_period_study(gateway_period_study, period_output)
    print('Computed cumulative 0→kT Gateway metrics and saved CSV.')
else:
    print('Reused cumulative Gateway metrics CSV:', period_output / 'gateway_period_metrics.csv')

period_rows = gateway_period_study['rows']
pairs = {
    k: {
        row['case']: row for row in period_rows if row['horizon_periods'] == k
    } for k in range(1, observer_period_count + 1)
}

def render_metrics_table(title, headers, records):
    """Keep comparisons side-by-side with one consistently styled row per horizon."""
    heading = ''.join(
        '<th style="padding:9px 12px;text-align:right;background:#eef3f8;'
        'border-bottom:1px solid #cbd5e1;white-space:normal;">'
        + h + '</th>' for h in headers)
    body = ''.join(
        '<tr style="background:' + ('#f6f8fc' if i % 2 else 'white') + '">'
        + ''.join(
            '<td style="text-align:right;padding:7px 12px;border-bottom:'
            '1px solid #e5e9f0;white-space:nowrap;">' + str(v) + '</td>'
            for v in row)
        + '</tr>' for i, row in enumerate(records))
    return (
        '<div style="margin:20px 0 8px;font:600 14px system-ui,sans-serif;">'
        + title + '</div><div style="overflow-x:auto;max-height:570px;'
        'border:1px solid #d5dde7;border-radius:8px;">'
        '<table style="border-collapse:collapse;font:12px system-ui,sans-serif;'
        'font-variant-numeric:tabular-nums;width:100%;min-width:850px;">'
        '<thead><tr>' + heading + '</tr></thead><tbody>' + body
        + '</tbody></table></div>'
    )

geometry_rows = []
tracking_rows = []
for k in range(1, observer_period_count + 1):
    perfect = pairs[k]['perfect']
    lso = pairs[k]['lso_estimated']
    horizon = f'0 → {k}T'
    geometry_rows.append([
        horizon,
        f"{perfect['terminal_observer_departure_km']:,.1f}",
        f"{perfect['max_observer_departure_km']:,.1f}",
        f"{perfect['terminal_gateway_range_km']:,.0f}",
        f"{perfect['new_gateway_observations']:,} / {perfect['gateway_observations']:,}",
        f"{perfect['new_lso_observations']:,} / {perfect['lso_observations']:,}",
    ])
    tracking_rows.append([
        horizon,
        f"{perfect['position_rmse_km']:.3f}",
        f"{lso['position_rmse_km']:.3f}",
        f"{perfect['velocity_rmse_m_s']:.4f}",
        f"{lso['velocity_rmse_m_s']:.4f}",
        f"{perfect['final_position_forecast_rss_sigma_km']:.3f}",
        f"{lso['final_position_forecast_rss_sigma_km']:.3f}",
    ])

display(HTML(render_metrics_table(
    'Observer geometry and accepted observations',
    ['Cumulative horizon', 'Departure at kT (km)', 'Max departure so far (km)',
     'Gateway range at kT (km)', 'Gateway obs. latest / total',
     'LSO obs. latest / total'], geometry_rows)))
display(HTML(render_metrics_table(
    'Gateway estimation over the complete 0 → kT history',
    ['Cumulative horizon', 'Position RMSE exact obs. (km)',
     'Position RMSE LSO obs. (km)', 'Velocity RMSE exact obs. (m/s)',
     'Velocity RMSE LSO obs. (m/s)',
     'Final pos. RSS σ exact obs. (km)', 'Final pos. RSS σ LSO obs. (km)'],
    tracking_rows)))
display(Markdown(
    '**Interpretation:** “Departure” is from the repeated rounded halo reference, '
    'whereas “Gateway range” is the actual observer–Gateway separation. '
    'Observation counts are shown as **new in the latest period / cumulative total**. '
    'RMSE includes every sample from time zero. Low cumulative RMSE when '
    'new observations are zero is prediction accuracy, **not active tracking**.'
))


## System identification — step 1: generate data

**Goal:** learn a local, unforced state-transition model that predicts the observer's next position and velocity. No model is fitted in this step.

Generate 20 training, 5 validation and 5 held-out test arcs. Each spans one nominal period, sampled uniformly every 30 minutes. Initial states are independently perturbed about the supplied state with 1 km and 0.01 m/s standard deviation per rotating-frame axis. These describe the local training domain, separately from the EKF's estimation prior.

Each file contains normalized rotating truth states, physical inertial position/velocity/acceleration, ideal angles, noisy angles and per-site visibility. Measured angles are NaN when unavailable. Training targets are the next normalized six-component state; the current state is the input. Truth states are simulation labels, not LSO position measurements.

Split by whole trajectory before constructing lag windows. Fit normalization using training data only; use validation for model selection and test data once for final assessment. Do not use invisible ideal angles as observations. The current dataset covers nearby initial conditions at one nominal phase, with no thrust or model discrepancy; it does not yet support stationkeeping identification.

In [ ]:
from halo_tracking.identification import generate_dataset
identification_summary = generate_dataset(cfg, ROOT / 'results/system_identification')
headers = ['Split', 'Arcs', 'Samples / arc', 'Transitions', 'Network visibility']
rows = ''.join('<tr>' + ''.join(f'<td>{value}</td>' for value in
    [name, item['arcs'], item['samples_per_arc'], item['transitions'], f"{100*item['visible_fraction']:.1f}%"])
    + '</tr>' for name, item in identification_summary.items())
display(HTML('<table><tr>' + ''.join(f'<th>{h}</th>' for h in headers) + '</tr>' + rows + '</table>'))